# 실습 보강: Flide 사용법: vLLM vs SGLang 속도 주장 확인하기

위에서부터 차례로 실행해 보세요.

### 설치와 환경 확인 (flide 0.1.10 고정)

In [ ]:
%%bash
nvidia-smi --query-gpu=name,memory.total,compute_cap --format=csv
pip install -q uv
uv pip install --system flide torchvision
uv pip install --system --no-deps flide==0.1.10
mkdir -p /content/outputs

### 타일 패치 없이 Flide 서빙 + 측정 (L4)

In [ ]:
import subprocess,time,urllib.request,urllib.error
subprocess.Popen("nohup flide serve --model Qwen/Qwen3.5-4B --dtype float16 --host 127.0.0.1 --port 8000 --max-request-tokens 6144 --max-branch-tokens 4096 > /content/flide4b.log 2>&1 &",shell=True)
t=time.time()
while time.time()-t<1100:
    try:
        urllib.request.urlopen('http://127.0.0.1:8000/v1/models',timeout=2);break
    except urllib.error.HTTPError: break
    except Exception: time.sleep(5)
print('startup %.0fs'%(time.time()-t))
print(open('/content/flide4b.log').read()[-600:])
r=subprocess.run("cd /content && python bench.py flide 8000 '' 7",shell=True,capture_output=True,text=True)
print(r.stdout,r.stderr[-1500:])

### vLLM·SGLang 설치

In [ ]:
%%bash
uv pip install --system vllm==0.33.0
curl -sSf https://sh.rustup.rs | sh -s -- -y --profile minimal
uv venv -p 3.12 /content/sv
. $HOME/.cargo/env; uv pip install --python /content/sv/bin/python 'sglang[all]==0.5.23'

### 벤치마크 스크립트 만들기

In [ ]:
%%bash
cat > /content/bench.py <<'E'
import sys,json,time,random,statistics,urllib.request
engine,port,model=sys.argv[1],int(sys.argv[2]),sys.argv[3]
REPS=int(sys.argv[4]) if len(sys.argv)>4 else 7
WARM=2
random.seed(7)
names=['Alder','Brix','Corvane','Dunmore','Elsing','Farrow','Gantry','Holloway','Ivers','Jessamy','Kellan','Lowmere','Marrow','Nettle','Orlin','Pellam','Quill','Rowan','Sedge','Tarn']
trades=['salt','wool','copper','timber','glass','honey','tin','silk','barley','amber','slate','flax','cider','iron','pearl','hemp','clay','wax','coal','lime']
out=[];FACTS=[]
for i,n in enumerate(names):
    year=random.choice(range(1100,1900));pop=random.choice(range(2,90))*100;t=trades[i]
    FACTS.append((n,year,pop,t))
    out.append(f'The town of {n} was founded in {year} on the river bank. Its main trade has always been {t}, which merchants carried to the coast by cart and barge. By the last census {n} had {pop} residents, most of whom worked in the {t} guild halls or on the surrounding farms. Travellers describe the market square of {n} as crowded in spring, when the {t} caravans arrive, and quiet in winter, when the roads are closed by snow. The council of {n} meets every second Tuesday to settle disputes about weights, tolls and the upkeep of the old stone bridge, and its minutes are kept in the church tower.')
TEXT='\n\n'.join(out)
def questions(n):
    qs={}
    for i in range(n):
        name,year,pop,t=FACTS[i]
        opts=[t]+random.Random(i).sample([x for x in trades if x!=t],3)
        random.Random(100+i).shuffle(opts)
        qs[f'q{i+1}']={'type':'choice','instructions':f'According to the passage, what is the main trade of the town of {name}?','criteria':{o:o.capitalize() for o in opts}}
    return qs
def post(b):
    r=urllib.request.Request(f'http://127.0.0.1:{port}/v1/systemone',json.dumps(b).encode(),{'Content-Type':'application/json'})
    t=time.perf_counter();raw=urllib.request.urlopen(r,timeout=600).read();return time.perf_counter()-t,json.loads(raw)
res={'engine':engine,'levels':{}}
for n in [1,5,10,20]:
    lat=[];answers=[]
    for rep in range(WARM+REPS):
        b={'state':f'[request {rep:04d}-{n}]\n\n'+TEXT,'questions':questions(n)}
        if model: b['model']=model
        dt,o=post(b)
        if rep>=WARM: lat.append(dt);answers.append({'in_tokens':o['usage']['input_tokens'],'answers':o['answers']})
    res['levels'][n]={'lat':lat,'median':statistics.median(lat),'answers':answers}
    print(engine,n,'median %.3fs'%statistics.median(lat),'in_tokens',answers[0]['in_tokens'],flush=True)
json.dump(res,open(f'/content/outputs/bench_{engine}.json','w'))
E

### vLLM 캐시 켬/끔 측정

In [ ]:
import subprocess,time,urllib.request
subprocess.run("pkill -f '[f]lide serve'",shell=True); time.sleep(10)
def serve_vllm(flag,log):
    subprocess.Popen("""nohup vllm serve Qwen/Qwen3.5-4B --dtype half --max-model-len 8192 --port 8001 --gpu-memory-utilization 0.80 %s --limit-mm-per-prompt '{"image":0,"video":0}' --max-num-seqs 16 --max-num-batched-tokens 4096 --max-cudagraph-capture-size 16 > %s 2>&1 &"""%(flag,log),shell=True)
    t=time.time()
    while time.time()-t<900:
        try: urllib.request.urlopen('http://127.0.0.1:8001/v1/models',timeout=2);break
        except Exception: time.sleep(5)
    print('startup %.0fs'%(time.time()-t))
serve_vllm('--enable-prefix-caching','/content/vllm.log')
r=subprocess.run("cd /content && python bench.py vllmcache 8001 Qwen/Qwen3.5-4B 7",shell=True,capture_output=True,text=True)
print(r.stdout,r.stderr[-1500:])
subprocess.run("pkill -f '[v]llm serve'",shell=True); time.sleep(15)
serve_vllm('--no-enable-prefix-caching','/content/vllm2.log')
r=subprocess.run("cd /content && python bench.py vllmnocache 8001 Qwen/Qwen3.5-4B 7",shell=True,capture_output=True,text=True)
print(r.stdout,r.stderr[-1500:])
subprocess.run("pkill -f '[v]llm serve'",shell=True)

### SGLang radix 끔/켬 측정 (백그라운드 실행 후 대기)

In [ ]:
import subprocess,time,os
open('/content/sg.sh','w').write('''
run(){
 nohup /content/sv/bin/python -m sglang.launch_server --model-path Qwen/Qwen3.5-4B --dtype $2 --port 8002 --context-length 8192 --mem-fraction-static 0.8 --attention-backend triton --max-running-requests 16 --cuda-graph-max-bs-decode 16 $3 > /content/sg_$1.log 2>&1 &
 for i in $(seq 1 180); do curl -s -o /dev/null http://127.0.0.1:8002/v1/models && break; sleep 5; done
 if curl -s -o /dev/null http://127.0.0.1:8002/v1/models; then cd /content && python bench.py $1 8002 Qwen/Qwen3.5-4B 7 >> /content/sg_bench.log 2>&1; else echo "FAILED $1 $2" >> /content/sg_bench.log; fi
 pkill -f '[s]glang.launch_server'; sleep 15
}
run sglang half --disable-radix-cache
run sglangradix half ""
if [ ! -f /content/outputs/bench_sglangradix.json ]; then run sglangradix bfloat16 ""; fi
echo ALL_DONE >> /content/sg_bench.log
''')
subprocess.Popen("nohup bash /content/sg.sh > /content/sg_main.log 2>&1 &",shell=True)
t=time.time()
while time.time()-t<1500:
    l=open('/content/sg_bench.log').read() if os.path.exists('/content/sg_bench.log') else ''
    if 'ALL_DONE' in l: break
    time.sleep(10)
print(l[-2500:])

### 엔진별 선택지 확률 비교 (TV distance, 1위 일치)

In [ ]:
import json, statistics as st
K=['flide','vllmcache','vllmnocache','sglang','sglangradix']
D={k:json.load(open(f'/content/outputs/bench_{k}.json'))['levels'] for k in K}
n='20'
for other in K[1:]:
    tv=[];ag=tot=0
    for rep in range(7):
        A=D['flide'][n]['answers'][rep]['answers'];B=D[other][n]['answers'][rep]['answers']
        for q,v in A.items():
            p=v['probabilities'];r=B[q]['probabilities']
            tv.append(0.5*sum(abs(p[x]-r[x]) for x in p));ag+=max(p,key=p.get)==max(r,key=r.get);tot+=1
    print('flide vs',other,'mean TV %.5f max TV %.5f argmax agree %d/%d'%(st.mean(tv),max(tv),ag,tot))

### T4 vs L4 비교 그래프

In [ ]:
import json
import matplotlib.pyplot as plt
K=['flide','vllmcache','vllmnocache','sglang','sglangradix']
N=['1','5','10','20']
D={k:json.load(open(f'/content/outputs/bench_{k}.json'))['levels'] for k in K}
L4={k:[D[k][n]['median'] for n in N] for k in K}
T4={'flide':[5.72,6.40,7.20,8.94],'vllmcache':[2.45,3.90,5.36,8.20],'vllmnocache':[2.57,12.72,25.30,50.61],'sglang':[4.76,24.14,48.27,96.60]}  # 공개 글의 T4 측정값
lab={'flide':'Flide','vllmcache':'vLLM (cache on)','vllmnocache':'vLLM (cache off)','sglang':'SGLang (radix off)','sglangradix':'SGLang (radix on, bf16)'}
fig,ax=plt.subplots(1,2,figsize=(13,5))
for a,data,title in [(ax[0],T4,'T4'),(ax[1],L4,'L4')]:
    for k,v in data.items(): a.plot([1,5,10,20],v,marker='o',label=lab[k],lw=3 if k=='flide' else 1.5)
    a.set_title(title);a.set_xlabel('questions per request');a.set_ylabel('median latency (s)');a.set_xticks([1,5,10,20]);a.grid(alpha=.3);a.legend(fontsize=8)
ax[0].set_ylim(0,100);ax[1].set_ylim(0,11)
plt.tight_layout();plt.savefig('/content/outputs/l4_vs_t4.png',dpi=130);plt.show()